# Phase 5: back-projection fusion validation

This notebook tests an automatic per-image NEDI–IMDN mixture.

For each validation image:

1. Shrink the NEDI output back to LR size.
2. Shrink the IMDN output back to LR size.
3. Calculate the mixture that most closely reproduces the original LR image.
4. Measure that fused HR image against the reference HR image.

The HR reference is used only for final measurement. It is never used to calculate the fusion percentage. The notebook reuses the cached outputs from notebook 14, so it does not rerun NEDI or IMDN and does not need a GPU.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository and dependencies ready.')


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CACHE_ROOT = DATA_ROOT / 'results' / 'phase5' / 'weight_selection_global_v1' / 'reconstruction_cache'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase5' / 'back_projection_validation_v1'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
METRICS_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

VALIDATION_IDS = ('0801', '0802', '0803', '0804', '0805')
SCALES = (2, 3, 4)
print('Input cache:', CACHE_ROOT)
print('Output folder:', OUTPUT_ROOT)


## Load the 15 cached validation cases

The notebook stops with a clear message if any cached file from notebook 14 is missing.


In [ ]:
from app.evaluation.images import load_rgb_image

cases = []
required_names = ('reference_hr.png', 'input_lr.png', 'nedi.png', 'imdn.png')
for scale in SCALES:
    for image_id in VALIDATION_IDS:
        case_root = CACHE_ROOT / f'x{scale}' / image_id
        missing = [name for name in required_names if not (case_root / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f'Missing cached files for {image_id} x{scale}: {missing}. '
                'Run notebook 14 completely first.'
            )
        reference_hr = load_rgb_image(case_root / 'reference_hr.png')
        input_lr = load_rgb_image(case_root / 'input_lr.png')
        nedi = load_rgb_image(case_root / 'nedi.png')
        imdn = load_rgb_image(case_root / 'imdn.png')
        if reference_hr.size != nedi.size or reference_hr.size != imdn.size:
            raise ValueError(f'Cached output sizes do not match for {image_id} x{scale}.')
        cases.append((image_id, scale, reference_hr, input_lr, nedi, imdn))

if len(cases) != 15:
    raise RuntimeError(f'Expected 15 validation cases; loaded {len(cases)}.')
print('Loaded all', len(cases), 'cached validation cases.')


## Calculate one automatic mixture per image

A higher IMDN weight means that the calculated result uses more IMDN. The percentages are not chosen using the HR image and can differ from one input image to another.


In [ ]:
from app.evaluation.experiment import write_results_csv
from app.evaluation.metrics import calculate_quality_metrics
from app.fusion.back_projection import evaluate_back_projection_fusion

records = []
for image_id, scale, reference_hr, input_lr, nedi, imdn in cases:
    fusion = evaluate_back_projection_fusion(
        reference_hr, input_lr, nedi, imdn, scale
    )
    nedi_metrics = calculate_quality_metrics(reference_hr, nedi, border=scale)
    imdn_metrics = calculate_quality_metrics(reference_hr, imdn, border=scale)
    record = {
        'validation_dataset': 'DIV2K_valid',
        'image': f'{image_id}.png',
        'scale': f'x{scale}',
        'weight_source': 'least_squares_match_to_original_lr',
        'degradation_check': 'bicubic_downsample_reconstruction_to_original_lr_size',
        'metric_border_pixels': scale,
        **fusion,
        **{f'nedi_{name}': value for name, value in nedi_metrics.items()},
        **{f'imdn_{name}': value for name, value in imdn_metrics.items()},
    }
    record['psnr_y_change_vs_imdn'] = record['psnr_y'] - record['imdn_psnr_y']
    record['ssim_y_change_vs_imdn'] = record['ssim_y'] - record['imdn_ssim_y']
    records.append(record)
    print(
        f"{image_id}.png x{scale}: IMDN {record['imdn_weight']:.2%}, "
        f"NEDI {record['nedi_weight']:.2%}, "
        f"PSNR-Y change {record['psnr_y_change_vs_imdn']:+.5f} dB"
    )

all_csv = write_results_csv(
    records,
    METRICS_ROOT / 'back_projection_validation_all.csv',
    overwrite=True,
)
print('Saved:', all_csv)


## Summarise and decide

The method succeeds only if its overall mean PSNR-Y is higher than IMDN alone. Results are also shown separately for x2, x3, and x4.


In [ ]:
import json
from datetime import UTC, datetime
from statistics import mean


def make_summary(label, group):
    return {
        'scale': label,
        'sample_count': len(group),
        'mean_imdn_weight': mean(float(row['imdn_weight']) for row in group),
        'mean_nedi_weight': mean(float(row['nedi_weight']) for row in group),
        'minimum_imdn_weight': min(float(row['imdn_weight']) for row in group),
        'maximum_imdn_weight': max(float(row['imdn_weight']) for row in group),
        'mean_fusion_psnr_y': mean(float(row['psnr_y']) for row in group),
        'mean_imdn_psnr_y': mean(float(row['imdn_psnr_y']) for row in group),
        'mean_psnr_y_change_vs_imdn': mean(float(row['psnr_y_change_vs_imdn']) for row in group),
        'mean_fusion_ssim_y': mean(float(row['ssim_y']) for row in group),
        'mean_imdn_ssim_y': mean(float(row['imdn_ssim_y']) for row in group),
        'mean_ssim_y_change_vs_imdn': mean(float(row['ssim_y_change_vs_imdn']) for row in group),
        'fusion_psnr_y_win_count': sum(float(row['psnr_y_change_vs_imdn']) > 0 for row in group),
    }


summary_records = [
    make_summary(f'x{scale}', [row for row in records if row['scale'] == f'x{scale}'])
    for scale in SCALES
]
overall = make_summary('overall', records)
summary_records.append(overall)
summary_csv = write_results_csv(
    summary_records,
    METRICS_ROOT / 'back_projection_summary.csv',
    overwrite=True,
)
improved = overall['mean_psnr_y_change_vs_imdn'] > 0
verdict = {
    'method': 'back_projection_nedi_imdn_fusion',
    'weight_calculation': 'per_image_least_squares_match_to_original_lr_clipped_0_to_1',
    'uses_hr_to_choose_weight': False,
    'improved_over_imdn': improved,
    **overall,
    'validation_dataset': 'DIV2K_valid',
    'validation_image_ids': list(VALIDATION_IDS),
    'validation_scales': list(SCALES),
    'generated_at_utc': datetime.now(UTC).isoformat(),
}
verdict_json = METRICS_ROOT / 'back_projection_verdict.json'
verdict_json.write_text(json.dumps(verdict, indent=2) + '\n', encoding='utf-8')

print('\nImproved over IMDN:', improved)
print(f"Mean IMDN weight: {overall['mean_imdn_weight']:.2%}")
print(f"Mean NEDI weight: {overall['mean_nedi_weight']:.2%}")
print(f"Mean PSNR-Y change: {overall['mean_psnr_y_change_vs_imdn']:+.6f} dB")
print(f"Mean SSIM-Y change: {overall['mean_ssim_y_change_vs_imdn']:+.6f}")
print('Saved:', summary_csv)
print('Saved:', verdict_json)


In [ ]:
import matplotlib.pyplot as plt

labels = [f"{row['image']} {row['scale']}" for row in records]
changes = [float(row['psnr_y_change_vs_imdn']) for row in records]
colours = ['#1769aa' if change > 0 else '#c44e52' for change in changes]

figure, axis = plt.subplots(figsize=(10, 6))
axis.barh(range(len(labels)), changes, color=colours)
axis.set_yticks(range(len(labels)), labels=labels)
axis.invert_yaxis()
axis.axvline(0, color='black', linewidth=1)
axis.set_xlabel('PSNR-Y change compared with IMDN alone (dB)')
axis.set_title('Back-projection fusion validation')
axis.grid(axis='x', alpha=0.25)
figure.tight_layout()
figure_path = FIGURE_ROOT / 'back_projection_psnr_change_vs_imdn.png'
figure.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


## Return the results

Download and send back these three files from **MyDrive/FYP_SR_Data/results/phase5/back_projection_validation_v1/metrics/**:

1. **back_projection_validation_all.csv**
2. **back_projection_summary.csv**
3. **back_projection_verdict.json**

This is the final fusion check. If it does not improve the validation PSNR-Y, Phase 5 will report that none of the three transparent fusion methods improved IMDN.
